In [4]:
import pandas as pd
df = pd.read_parquet("../../../artifacts/cache/fox/groups/fixed_window/alert_groups/alert_groups_base.parquet", engine="pyarrow")


In [5]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("shape:", df.shape)
df.dtypes

shape: (10310, 12)


alert_group_id        object
group_label           object
n_alerts               int64
weight               float64
duration_sec           int64
n_items                int64
n_hosts                int64
n_shorts               int64
n_sigs                 int64
n_internal_ips         int64
n_external_ips         int64
alerts_per_second    float64
dtype: object

### Distribution of baseline features

In [6]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
alert_group_id,10310,10310,fixed_window:821310059,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
group_label,10310,2,benign,9498,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_alerts,10310.0,NaN,NaN,NaN,45.887876,166.218512,1.0,3.0,3.0,6.0,1527.0
weight,10310.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
duration_sec,10310.0,NaN,NaN,NaN,0.14937,0.35647,0.0,0.0,0.0,0.0,1.0
n_items,10310.0,NaN,NaN,NaN,4.927255,2.418914,2.0,4.0,4.0,4.0,25.0
n_hosts,10310.0,NaN,NaN,NaN,1.092726,0.31691,1.0,1.0,1.0,1.0,5.0
n_shorts,10310.0,NaN,NaN,NaN,1.444423,1.157902,1.0,1.0,1.0,1.0,12.0
n_sigs,10310.0,NaN,NaN,NaN,2.390107,1.187161,0.0,2.0,2.0,2.0,12.0
n_internal_ips,10310.0,NaN,NaN,NaN,1.092726,0.31691,1.0,1.0,1.0,1.0,5.0


In [7]:
print("group_label counts:")
print(df["group_label"].value_counts(dropna=False))

print("\nn_external_ips value counts:")
print(df["n_external_ips"].value_counts())

group_label counts:
group_label
benign    9498
attack     812
Name: count, dtype: int64

n_external_ips value counts:
n_external_ips
0    10310
Name: count, dtype: int64


**Note:** `n_external_ips` is 0 for every group here and `n_internal_ips` == `n_hosts` exactly. AIT-ADS alert IPs are all RFC1918 private addresses (confirmed directly against `data/alerts_csv/*.txt` — zero public IPs across all 8 scenarios), so the internal/external IP split computed by the baseline features is a dead column on this dataset. It's presumably meaningful on the `cscas` (Suricata/`ExtIP`) dataset instead.

### Constant columns (same value for every group)

In [8]:
nunique = df.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()

print(f"{len(constant_cols)} / {len(df.columns)} columns are constant across all {len(df)} groups:")
for col in constant_cols:
    print(f"  {col!r}: {df[col].iloc[0]!r}")

2 / 12 columns are constant across all 10310 groups:
  'weight': np.float64(1.0)
  'n_external_ips': np.int64(0)
